# exchange-engine GPU analytics on Colab (fallback runner)

Self-contained fallback for `remote/kaggle.py`: run the same benchmark suite on a Colab T4.
Runtime > Change runtime type > T4 GPU, then run the cells in order. Results from this notebook are
labelled with the archive's own `source`; only a real archive on a T4 can ever reach the documents.

In [ ]:
# 1. T4 only: stop here on any other card.
import subprocess, sys
line = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print(line)
if "T4" not in line:
    raise SystemExit("This notebook only runs on an NVIDIA T4 (got: %r)" % line)

In [ ]:
# 2. The repository and its dependencies.
!git clone -q --branch gpu-analytics https://github.com/Tanneywanney25/async-matching-engine.git repo
%cd repo
!pip install -q zstandard sortedcontainers hypothesis
!nvcc --version | tail -1

## 3. An archive

Either download the private Kaggle dataset (upload your own `access_token` from Kaggle Settings > API
when prompted) or generate a synthetic archive with the identical schema.

In [ ]:
USE_KAGGLE = False  # set True to download tanushbhatia/exchange-engine-archive with your token
if USE_KAGGLE:
    from google.colab import files
    import os
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    up = files.upload()  # choose your access_token file
    name = next(iter(up))
    open(os.path.expanduser("~/.kaggle/access_token"), "wb").write(up[name])
    os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
    !pip install -q "kaggle>=1.8.0"
    !kaggle datasets download -d tanushbhatia/exchange-engine-archive -p data/archive --unzip
    ARCHIVE, SOURCE = "data/archive", "real"
else:
    !python scripts/synth_archive.py --out data/synth --snapshots 300000 --trades 100000
    ARCHIVE, SOURCE = "data/synth", "synthetic"
print(ARCHIVE, SOURCE)

In [ ]:
# 4. Every benchmark section, short version (drop --quick for the full run, about 30 minutes).
!python -m bench.run_all --archive {ARCHIVE} --out results/colab --source {SOURCE} --quick

In [ ]:
# 5. Download the results.
!cd results && zip -qr ../colab_results.zip colab
from google.colab import files
files.download("colab_results.zip")